# TimesFM 2.5 Training - T4 GPU FIXED VERSION

**All syntax errors fixed, memory optimizations applied**

---

## 🎯 What's Fixed:
- ✅ Syntax errors in notebook cells
- ✅ Unterminated string literals
- ✅ Memory optimizations for T4 GPU
- ✅ Proper code formatting

## 📊 Strategy:
Memory optimizations to fit TimesFM 2.5 200M on 14.56GB T4

## Step 1: Check GPU

In [ ]:
!nvidia-smi

## Step 2: Install & Setup

In [ ]:
# Uninstall problematic torchao
!pip uninstall -y torchao

# Install dependencies
!pip install -q transformers peft torch pandas numpy pyyaml accelerate

# Setup memory management
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Memory: {gpu_mem:.1f} GB")

## Step 3: Clone Repository

In [ ]:
# Clone repository
!git clone https://github.com/ntquy9901/stockvoli-research.git

# Change directory
import os
os.chdir('stockvoli-research')

print(f"Working directory: {os.getcwd()}")
!git pull origin master

# Create directories
!mkdir -p experiments models

print("Repository ready!")

## Step 4: Create Optimized Config

In [ ]:
import yaml

# Create memory-optimized config
optimized_config = {
    'model': {
        'name': 'timesfm-2.5-200m',
        'context_length': 64,
        'prediction_length': 1,
        'dtype': 'bfloat16'
    },
    'training': {
        'batch_size': 1,
        'gradient_accumulation_steps': 8,
        'learning_rate': 1e-4,
        'epochs': 100,
        'optimizer': 'SGD',
        'max_grad_norm': 1.0
    },
    'data': {
        'window_size': 90,
        'test_size': 0.2
    },
    'lora': {
        'r': 4,
        'lora_alpha': 8,
        'target_modules': 'all-linear',
        'lora_dropout': 0.05,
        'bias': 'none'
    },
    'experiment_tracking': {
        'experiments_dir': 'experiments',
        'models_dir': 'models',
        'log_level': 'INFO'
    }
}

# Save config
with open('configs/config_optimized.yaml', 'w') as f:
    yaml.dump(optimized_config, f)

# Copy to main config
import shutil
shutil.copy('configs/config_optimized.yaml', 'configs/config.yaml')

print('Memory-optimized config created!')
print('Batch size: 1 (accumulate to 8)')
print('Context length: 64 (reduced)')
print('Mixed precision: bfloat16')

## Step 5: Verify Data

In [ ]:
from pathlib import Path

processed_dir = Path('data/processed')
files = list(processed_dir.glob('*_processed.csv'))
print(f"Found {len(files)} stock files")

# Check memory
import torch
torch.cuda.empty_cache()

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nGPU Memory: {allocated:.1f}/{total:.1f} GB used")

## Step 6: Run Training (FIXED - No Syntax Errors)

In [ ]:
print('Starting TimesFM 2.5 training...')
print('\nOptimizations applied:')
print('- Gradient accumulation: batch=1, accumulate=8')
print('- Mixed precision: bfloat16')
print('- Context length: 64')
print('\nExpected duration: 5-6 hours')

# Run training script directly
!python src/model_training_fixed.py

## Step 7: Monitor Progress

In [ ]:
# Check logs
!tail -30 experiments/model_training.log

# Check models
from pathlib import Path
models_dir = Path('models')
if models_dir.exists():
    models = list(models_dir.glob('*.pt'))
    if models:
        print(f"\nFound {len(models)} model(s)")
        for m in models:
            print(f"  {m.name}")

## Troubleshooting

### Syntax Error
**FIXED:** All syntax errors resolved in this version

### CUDA OOM
If still OOM: Runtime → Restart → Run from Step 1

### Training Slow
Expected: 5-6 hours (gradient accumulation)